# 📊 stock_analyzer 코랩 시각화 노트북

로컬 `stock_analyzer` 시스템이 GitHub Pages(`textgun.github.io/stock-reports`)로 push한
분석 데이터를 불러와 시각화한다. **위에서부터 순서대로 실행**하면 된다.

| 섹션 | 데이터 소스 | 비고 |
|------|------------|------|
| 1. SWOT 점수 추이·비교 | `colab_data/history.json` | 자동 로드 |
| 2. 밸류에이션 분석 | `colab_data/company_summaries.json` | 자동 로드 |
| 3. 시세 캔들차트 | FinanceDataReader (코랩에서 직접 조회) | 자동 로드 |
| 4. 포트폴리오 성과 | `daily_pnl.json`·`realized_pnl.json` **수동 업로드** | 계좌 데이터는 공개 repo에 없음 |

데이터 갱신 주기: 평일 09:55 / 14:00 / 15:15 (push_dashboard cron에 편승).
`signals['date']` 출력으로 데이터 기준일을 확인할 것.

> ⚠️ 이 노트북은 로컬 `stock_analyzer/colab/` 원본이 push될 때마다 repo 버전이 덮인다.
> 코랩에서 수정해 유지하려면 **파일 → Drive에 사본 저장**으로 개인 사본을 만들 것.

## 셋업 — 패키지 설치·공통 스타일

In [ ]:
# 최초 1회 실행 — 코랩에 기본 포함되지 않은 패키지 설치
%pip install -q finance-datareader


In [ ]:
# 공통 임포트 + 차트 스타일 상수
import json

import pandas as pd
import plotly.graph_objects as go
import requests
from plotly.subplots import make_subplots

BASE_URL = "https://textgun.github.io/stock-reports/colab_data"

# 카테고리 팔레트 (고정 순서 — 8개 초과 시리즈는 그리지 않는다. 순환 재사용 금지)
PALETTE = ["#2a78d6", "#1baf7a", "#eda100", "#008300",
           "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"]
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
UP, DOWN = "#e34948", "#2a78d6"   # 국내 관례: 상승=빨강, 하락=파랑


def base_layout(fig, title="", height=None):
    fig.update_layout(
        title=title, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
        font=dict(color=INK2, size=12), title_font=dict(color=INK, size=15),
        margin=dict(l=60, r=30, t=60, b=40),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
    )
    if height:
        fig.update_layout(height=height)
    fig.update_xaxes(gridcolor=GRID, linecolor=AXIS, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=AXIS, zeroline=False)
    return fig


In [ ]:
# 데이터 로드 — GitHub Pages에 push된 분석 데이터 3종
def load(name):
    r = requests.get(f"{BASE_URL}/{name}", timeout=15)
    r.raise_for_status()
    return r.json()

summaries = load("company_summaries.json")   # 종목별 SWOT·밸류에이션 요약
history   = load("history.json")             # 종목별 점수 이력
signals   = load("trading_signals.json")     # 최근 매매 신호

print(f"종목 요약 {len(summaries)}개 / 점수 이력 {len(history)}개 / 신호 기준일 {signals['date']}")
print(f"시장 국면: {signals.get('market_phase')} / 주도 섹터: {list((signals.get('leading_sectors') or {}).keys())}")


## 1. SWOT 점수 추이·비교

`TICKERS`에 원하는 종목코드를 넣는다 (최대 8개 — 팔레트 슬롯 수).
기본값은 이력 데이터가 많은 순 상위 5개.

In [ ]:
# 이력 레코드가 많은 순으로 기본 5개 선택 — 원하면 직접 지정: TICKERS = ["005930", "236200"]
TICKERS = sorted(history, key=lambda t: len(history[t]["records"]), reverse=True)[:5]
TICKERS = TICKERS[:8]   # 팔레트 슬롯 수 초과 금지

fig = go.Figure()
for i, t in enumerate(TICKERS):
    df = pd.DataFrame(history[t]["records"])
    name = f"{history[t]['company_name']} ({t})"
    fig.add_trace(go.Scatter(
        x=df["date"], y=df["total_score"], mode="lines+markers", name=name,
        line=dict(color=PALETTE[i], width=2), marker=dict(size=7),
        hovertemplate="%{x}<br>" + name + " %{y}점<extra></extra>"))
base_layout(fig, "SWOT 종합 점수 추이", height=450)
fig.update_yaxes(title="종합 점수 (0~100)")
fig.update_layout(hovermode="x unified")
fig.show()


In [ ]:
# SWOT 레이더 — 섹터별 최대점수 대비 달성률(%)로 정규화해 서로 다른 섹터도 비교 가능
COMPARE = TICKERS[:4]
CATS = ["strength", "weakness", "opportunity", "threat"]
LABELS = ["Strength", "Weakness", "Opportunity", "Threat"]

fig = go.Figure()
for i, t in enumerate(COMPARE):
    s = summaries.get(t)
    if not s:
        print(f"  (요약 데이터 없음 — {t} 건너뜀)")
        continue
    bd, mx = s["breakdown"], s.get("sector_max") or {}
    vals = [bd[c] / max(mx.get(c, 25), 1) * 100 for c in CATS]
    fig.add_trace(go.Scatterpolar(
        r=vals + vals[:1], theta=LABELS + LABELS[:1],
        name=f"{s['company']} ({t})",
        line=dict(color=PALETTE[i], width=2)))   # 겹침 가독성 — fill 없이 선만
base_layout(fig, "SWOT 달성률 비교 (섹터 최대점수 대비 %)", height=480)
fig.update_layout(polar=dict(bgcolor=SURFACE,
                             radialaxis=dict(range=[0, 100], gridcolor=GRID),
                             angularaxis=dict(gridcolor=GRID)))
fig.show()


## 2. 밸류에이션 분석

`company_summaries.json`의 `valuation`(PER·PBR·PEG·ROE)과
`valuation_engine`(S-RIM·PER밴드·PBR-ROE 종합 적정가·존 판정)을 사용한다.
값은 **각 종목의 마지막 분석 시점 기준**이다 (`updated` 필드 참고).

In [ ]:
# 밸류에이션 DataFrame 구성 + 테이블 뷰
rows = []
for t, s in summaries.items():
    v, ve = s.get("valuation") or {}, s.get("valuation_engine") or {}
    rows.append(dict(
        ticker=t, company=s["company"], sector=s.get("sector"),
        updated=s.get("updated"), score=s.get("score"),
        per=v.get("per"), pbr=v.get("pbr"), peg=v.get("peg"), roe=v.get("roe"),
        fair=ve.get("fair_value"), zone=ve.get("zone"),
        gap=ve.get("gap_pct"), confidence=ve.get("confidence")))
val_df = pd.DataFrame(rows)
print(f"밸류에이션 데이터 보유: {val_df['pbr'].notna().sum()}개 / 존 판정 보유: {val_df['zone'].notna().sum()}개")
val_df.sort_values("gap", ascending=False).head(15)


In [ ]:
# ROE vs PBR 산점도 — 점선 위(적정 PBR = ROE/10 초과)면 ROE 대비 비싸게 거래되는 종목
d = val_df.dropna(subset=["roe", "pbr"]).query("0 < pbr < 15 and -50 < roe < 60")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=d["roe"], y=d["pbr"], mode="markers",
    marker=dict(size=10, color=PALETTE[0], opacity=0.75,
                line=dict(color=SURFACE, width=2)),
    name="종목", text=d["company"] + " (" + d["ticker"] + ")",
    hovertemplate="%{text}<br>ROE %{x:.1f}% · PBR %{y:.2f}배<extra></extra>"))
x_max = float(d["roe"].max())
fig.add_trace(go.Scatter(
    x=[0, x_max], y=[0, x_max / 10], mode="lines", name="적정 PBR = ROE/10",
    line=dict(color=MUTED, width=1.5, dash="dash")))
base_layout(fig, "ROE vs PBR (점선 아래 = justified PBR 대비 저평가)", height=500)
fig.update_xaxes(title="ROE (%)")
fig.update_yaxes(title="PBR (배)")
fig.show()


In [ ]:
# 적정가치 괴리율 상·하위 — 괴리율 = (적정가 - 당시 주가) / 당시 주가 (%)
# 신뢰도 LOW(방법 간 편차 큼)는 기본 제외 — 포함하려면 include_low=True
include_low = False
g = val_df.dropna(subset=["gap"])
if not include_low:
    g = g[g["confidence"] != "LOW"]
g = pd.concat([g.nlargest(12, "gap"), g.nsmallest(6, "gap")]).drop_duplicates("ticker")
g = g.sort_values("gap")
label = g["company"] + " (" + g["ticker"] + ")"

fig = go.Figure(go.Bar(
    x=g["gap"], y=label, orientation="h",
    marker=dict(color=[UP if v < 0 else DOWN for v in g["gap"]]),
    text=[f"{v:+.0f}%" for v in g["gap"]], textposition="outside",
    textfont=dict(color=INK2),
    customdata=g[["zone", "confidence"]],
    hovertemplate="%{y}<br>괴리율 %{x:+.1f}% · 존 %{customdata[0]} · 신뢰도 %{customdata[1]}<extra></extra>"))
base_layout(fig, "적정가치 괴리율 (파랑 = 저평가·상승여력 / 빨강 = 고평가)",
            height=max(420, 26 * len(g) + 120))
fig.update_xaxes(title="괴리율 (%)")
fig.add_vline(x=0, line_color=AXIS, line_width=1)
fig.show()


## 3. 시세 캔들차트

FinanceDataReader로 코랩에서 직접 시세를 받아 캔들 + MA20/60 + 거래량 + RSI(14)를 그린다.
`TICKER`와 `START`를 바꿔 재실행하면 된다.

In [ ]:
import FinanceDataReader as fdr

TICKER = "005930"
START = "2025-07-01"

px = fdr.DataReader(TICKER, START)
name = summaries.get(TICKER, {}).get("company", TICKER)

# 지표 계산
px["MA20"] = px["Close"].rolling(20).mean()
px["MA60"] = px["Close"].rolling(60).mean()
delta = px["Close"].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
px["RSI"] = 100 - 100 / (1 + gain / loss)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.58, 0.20, 0.22], vertical_spacing=0.03,
                    subplot_titles=("", "거래량", "RSI(14)"))
fig.add_trace(go.Candlestick(
    x=px.index, open=px["Open"], high=px["High"], low=px["Low"], close=px["Close"],
    name="시세", increasing_line_color=UP, increasing_fillcolor=UP,
    decreasing_line_color=DOWN, decreasing_fillcolor=DOWN), row=1, col=1)
fig.add_trace(go.Scatter(x=px.index, y=px["MA20"], name="MA20",
                         line=dict(color=PALETTE[2], width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=px.index, y=px["MA60"], name="MA60",
                         line=dict(color=PALETTE[4], width=1.5)), row=1, col=1)
vol_color = [UP if c >= o else DOWN for o, c in zip(px["Open"], px["Close"])]
fig.add_trace(go.Bar(x=px.index, y=px["Volume"], name="거래량",
                     marker=dict(color=vol_color), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=px.index, y=px["RSI"], name="RSI",
                         line=dict(color=PALETTE[0], width=1.5),
                         showlegend=False), row=3, col=1)
fig.add_hline(y=70, line_color=MUTED, line_dash="dash", line_width=1, row=3, col=1)
fig.add_hline(y=30, line_color=MUTED, line_dash="dash", line_width=1, row=3, col=1)

base_layout(fig, f"{name} ({TICKER}) — 캔들 · MA20/60 · 거래량 · RSI", height=760)
fig.update_layout(xaxis_rangeslider_visible=False)
fig.update_yaxes(title="원", row=1, col=1)
fig.update_yaxes(range=[0, 100], row=3, col=1)
fig.show()


## 4. 포트폴리오 성과 (선택 — 수동 업로드)

계좌 손익 데이터는 공개 repo에 올리지 않으므로, 로컬 `stock_analyzer/data/` 폴더의
**`daily_pnl.json`·`realized_pnl.json`** 을 아래 셀에서 직접 업로드한다.
업로드하지 않으면 이 섹션은 건너뛰어도 된다.

In [ ]:
# 파일 업로드 — daily_pnl.json, realized_pnl.json 선택 (둘 중 하나만 올려도 됨)
from google.colab import files

uploaded = files.upload()
daily = json.loads(uploaded["daily_pnl.json"]) if "daily_pnl.json" in uploaded else None
realized = json.loads(uploaded["realized_pnl.json"]) if "realized_pnl.json" in uploaded else None
print("daily_pnl:", "OK" if daily else "없음", "/ realized_pnl:", "OK" if realized else "없음")


In [ ]:
# 보유 종목별 미실현 손익률 (daily_pnl.json)
if daily and daily.get("details"):
    h = pd.DataFrame(daily["details"]).T.astype({"pnl": float}).sort_values("pnl")
    label = h["name"] + " (" + h.index + ")"
    fig = go.Figure(go.Bar(
        x=h["pnl"], y=label, orientation="h",
        marker=dict(color=[UP if v >= 0 else DOWN for v in h["pnl"]]),
        text=[f"{v:+.1f}%" for v in h["pnl"]], textposition="outside",
        textfont=dict(color=INK2),
        customdata=h[["qty", "avg", "cur"]],
        hovertemplate="%{y}<br>%{customdata[0]}주 · 매입 %{customdata[1]:,.0f}원 → 현재 %{customdata[2]:,.0f}원<extra></extra>"))
    base_layout(fig, f"보유 종목 미실현 손익률 — {daily['date']} 기준", height=max(320, 60 * len(h) + 140))
    fig.update_xaxes(title="손익률 (%)")
    fig.add_vline(x=0, line_color=AXIS, line_width=1)
    fig.show()
    print(f"당일 손익 {daily.get('day_pnl_won', 0):+,.0f}원 ({daily.get('pnl_pct', 0) * 100:+.2f}%) · "
          f"총자금 {daily.get('total_value', 0):,.0f}원 (누적 {daily.get('total_pnl_pct', 0):+.2f}%)")
elif daily:
    print(f"보유 종목 없음 (전량 현금화 상태) · 총자금 {daily.get('total_value', 0):,.0f}원 "
          f"(누적 {daily.get('total_pnl_pct', 0):+.2f}%)")
else:
    print("daily_pnl.json 미업로드 — 건너뜀")


In [ ]:
# 매도 사유별 누적 성과 (realized_pnl.json) — briefing.py의 '매도 사유별 누적 통계'와 같은 관점
if realized and realized.get("trades"):
    tr = pd.DataFrame(realized["trades"])
    tr["사유"] = (tr["reason"].fillna("(기록 없음)")
                  .str.split(" [", regex=False).str[0]
                  .str.split(" (", regex=False).str[0])
    agg = tr.groupby("사유").agg(
        건수=("ticker", "size"), 손익합_원=("pnl_won", "sum"),
        평균손익률=("pnl_pct", "mean"),
        승률=("pnl_won", lambda x: (x > 0).mean() * 100),
        평균슬리피지=("slippage_pct", "mean")).round(2)
    display(agg)

    tr = tr.sort_values("date")
    tr["누적실현손익"] = tr["pnl_won"].cumsum()
    fig = go.Figure(go.Scatter(
        x=tr["date"], y=tr["누적실현손익"], mode="lines+markers",
        line=dict(color=PALETTE[0], width=2), marker=dict(size=7),
        text=tr["name"] + " " + tr["reason"].fillna(""),
        hovertemplate="%{x} %{text}<br>누적 %{y:,.0f}원<extra></extra>"))
    base_layout(fig, f"누적 실현손익 (총 {realized['total_realized']:+,.0f}원)", height=420)
    fig.update_yaxes(title="원")
    fig.add_hline(y=0, line_color=AXIS, line_width=1)
    fig.show()
else:
    print("realized_pnl.json 미업로드 — 건너뜀")
